# 🏎️ Box Box Bot — Explore Notebook

> *"Box box, the agent has forgotten how to turn."*

This notebook is your **pre-season testing ground**. Use it to:
- Verify the environment and personalities are working
- Watch a random agent completely lose the plot
- Do short experimental training runs
- Visualise results and replay GIFs after full training

---
⚠️ **For real training runs, use the terminal:**
```bash
python agents/train.py --test   # verify pipeline first
python agents/train.py          # full championship training
```
Notebooks aren't ideal for long training runs — the kernel can drop and you lose everything.

---
## 🔧 1 — Imports

Run this first. `sys.path` is extended so the notebook can import directly from `agents/`.

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import imageio
from IPython.display import Image, display
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.callbacks import BaseCallback

# Allow imports from agents/
sys.path.append(os.path.join(os.getcwd(), 'agents'))
from personalities import F1DriverEnv

print('✅ Imports OK — engineers are in position.')

---
## 🏁 2 — Driver Lineup

Print the full driver lineup and their personality configs.

In [ ]:
print('\n🏎️  Box Box Bot — Driver Lineup\n')
for name, cfg in F1DriverEnv.PERSONALITIES.items():
    print(f"  {cfg['emoji']}  #{cfg['number']}  {name.replace('_', ' ').title()}")
    print(f"       {cfg['description']}")
    print(f"       speed_bonus={cfg['speed_bonus']}  "
          f"offtrack_penalty={cfg['offtrack_penalty']}  "
          f"smooth_bonus={cfg['smooth_bonus']}\n")

---
## 🌍 3 — Verify the Environment

Confirm CarRacing-v3 opens correctly and check the observation and action spaces.

- **Observation:** 96×96 RGB image — what the driver sees
- **Action:** `[steering, throttle, brake]` — all continuous

In [ ]:
import gymnasium as gym

env = gym.make('CarRacing-v3', continuous=True, render_mode='rgb_array')
obs, info = env.reset()

print(f'Observation shape : {obs.shape}  ← 96×96 RGB')
print(f'Action space      : {env.action_space}')
print(f'  steering        : {env.action_space.low[0]:.1f} to {env.action_space.high[0]:.1f}')
print(f'  throttle        : {env.action_space.low[1]:.1f} to {env.action_space.high[1]:.1f}')
print(f'  brake           : {env.action_space.low[2]:.1f} to {env.action_space.high[2]:.1f}')

env.close()
print('\n✅ Environment verified — car is in the garage.')

---
## 👀 4 — The Driver's Eye View

The agent learns purely from this 96×96 image. No GPS, no lap times, no engineer on the radio — just pixels.

In [ ]:
env = gym.make('CarRacing-v3', continuous=True, render_mode='rgb_array')
obs, _ = env.reset()
frame = env.render()
env.close()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(frame)
axes[0].set_title('What we see (rendered)', fontsize=12)
axes[0].axis('off')
axes[1].imshow(obs)
axes[1].set_title('What the agent sees (96×96 obs)', fontsize=12)
axes[1].axis('off')

plt.suptitle("Driver's Eye View — Pre-Season", fontsize=14, fontweight='bold')
plt.tight_layout()
os.makedirs('replays', exist_ok=True)
plt.savefig('replays/drivers_eye_view.png', dpi=120, bbox_inches='tight')
plt.show()
print('📸 Saved to replays/drivers_eye_view.png')

---
## 🤪 5 — The Unhinged Rookie (Random Agent)

Before any training, this is what pure chaos looks like.

Random action every step — no steering logic, no braking logic, nothing.
This is your **FP1 baseline**. The thing you're training away from.

In [ ]:
env = gym.make('CarRacing-v3', continuous=True, render_mode='rgb_array')
obs, _ = env.reset()

frames, total_reward = [], 0
MAX_STEPS = 400

for _ in range(MAX_STEPS):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    frames.append(env.render())
    if terminated or truncated:
        break

env.close()

os.makedirs('replays', exist_ok=True)
imageio.mimsave('replays/rookie_chaos.gif', frames, fps=30)

print(f'Total reward  : {total_reward:.2f}')
print(f'Steps survived: {len(frames)}')
print('📼 Saved to replays/rookie_chaos.gif')
display(Image(filename='replays/rookie_chaos.gif'))

---
## 🧬 6 — Personality Sanity Check

Verify all four driver personalities load and step correctly before committing to a long training run.

In [ ]:
print('Running personality sanity check...\n')

for name, cfg in F1DriverEnv.PERSONALITIES.items():
    env = F1DriverEnv(personality=name, render_mode='rgb_array')
    obs, _ = env.reset()
    action = env.action_space.sample()
    obs, reward, _, _, _ = env.step(action)
    env.close()
    print(f"  ✅  {cfg['emoji']}  #{cfg['number']}  {name:<20} step reward: {reward:.4f}")

print('\nAll personalities OK — ready for training.')

---
## 🏋️ 7 — Shakedown Training Run

A short 50k-step run on one driver — just enough to see the agent start learning.

This is **not** the real training run. It's a shakedown test.

> ⏱️ Roughly 5–10 minutes on a Mac.

Change `PERSONALITY` to try a different driver.

In [ ]:
class RewardLogger(BaseCallback):
    """Tracks per-episode rewards with one accumulator per parallel env."""

    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episode_rewards = []
        self._ep_reward_buffer = None

    def _on_training_start(self) -> None:
        self._ep_reward_buffer = [0.0] * self.training_env.num_envs

    def _on_step(self) -> bool:
        for i, (r, done) in enumerate(zip(self.locals['rewards'], self.locals['dones'])):
            self._ep_reward_buffer[i] += r
            if done:
                self.episode_rewards.append(self._ep_reward_buffer[i])
                self._ep_reward_buffer[i] = 0.0
        return True


PERSONALITY = 'sunday_specialist'  # ← change me
TIMESTEPS   = 50_000

vec_env  = make_vec_env(lambda: F1DriverEnv(personality=PERSONALITY), n_envs=4)
callback = RewardLogger()

model = PPO(
    'CnnPolicy', vec_env,
    verbose=0, learning_rate=3e-4, n_steps=512, batch_size=128,
)

print(f'🚦 Shakedown — training {PERSONALITY} for {TIMESTEPS:,} steps...')
model.learn(total_timesteps=TIMESTEPS, callback=callback)

os.makedirs('models', exist_ok=True)
model.save(f'models/{PERSONALITY}_{TIMESTEPS}')
vec_env.close()

print(f'\n✅ Done — model saved to models/{PERSONALITY}_{TIMESTEPS}.zip')

---
## 📈 8 — Learning Curve

Plot the reward curve from the shakedown run.

With 50k steps it'll be noisy — that's expected. Look for the **direction of travel**, not the absolute numbers.

In [ ]:
rewards = callback.episode_rewards

if not rewards:
    print('No episodes completed — try more timesteps.')
else:
    cfg     = F1DriverEnv.PERSONALITIES[PERSONALITY]
    colour  = cfg['colour']
    window  = max(1, len(rewards) // 10)
    smoothed = pd.Series(rewards).rolling(window, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(rewards, alpha=0.25, color=colour, label='Raw episode reward')
    ax.plot(smoothed, color=colour, linewidth=2, label=f'Rolling avg (window={window})')
    ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
    ax.set_title(
        f"{cfg['emoji']}  #{cfg['number']}  {PERSONALITY.replace('_', ' ').title()} — Learning Curve",
        fontsize=14, fontweight='bold'
    )
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total Reward')
    ax.legend()
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(f'replays/{PERSONALITY}_learning_curve.png', dpi=120, bbox_inches='tight')
    plt.show()

    print(f'Episodes completed : {len(rewards)}')
    print(f'Final avg reward   : {np.mean(rewards[-10:]):.2f}  (last 10 episodes)')
    print(f'📸 Saved to replays/{PERSONALITY}_learning_curve.png')

---
## 🎬 9 — Record a Replay GIF

Load the shakedown model and capture one episode as a GIF.

Even after 50k steps it won't be pretty — but you should see it behaving slightly less chaotically than the random agent.

In [ ]:
model = PPO.load(f'models/{PERSONALITY}_{TIMESTEPS}')
env   = F1DriverEnv(personality=PERSONALITY, render_mode='rgb_array')
obs, _ = env.reset()

frames, total_reward, steers = [], 0.0, []

for _ in range(600):
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward
    steers.append(float(abs(action[0])))
    frames.append(env.render())
    if terminated or truncated:
        break

env.close()

gif_path = f'replays/{PERSONALITY}_{TIMESTEPS}_shakedown.gif'
imageio.mimsave(gif_path, frames, fps=30)

print('📊 Shakedown debrief:')
print(f'   Total reward  : {round(total_reward, 2)}')
print(f'   Steps         : {len(frames)}')
print(f'   Avg steering  : {round(float(np.mean(steers)), 4)}')
print(f'\n📼 Saved to {gif_path}')
display(Image(filename=gif_path))

---
## 📊 10 — Compare All Trained Drivers

Run this **after** full training (`python agents/train.py`) and evaluation (`python agents/evaluate.py`).

Loads the `results/evaluation_summary.csv` and plots a comparison across all four drivers.

In [ ]:
summary_path = 'results/evaluation_summary.csv'

if not os.path.exists(summary_path):
    print('No evaluation summary found yet.')
    print('Run: python agents/evaluate.py')
else:
    df = pd.read_csv(summary_path)

    fig, axes = plt.subplots(1, 3, figsize=(14, 5))

    colours = [
        F1DriverEnv.PERSONALITIES[d]['colour']
        for d in df['driver']
        if d in F1DriverEnv.PERSONALITIES
    ]

    # Total reward
    axes[0].barh(df['driver'], df['total_reward'], color=colours)
    axes[0].set_title('Total Reward', fontweight='bold')
    axes[0].axvline(0, color='gray', linewidth=0.8)
    axes[0].invert_yaxis()

    # Avg steering (lower = smoother)
    axes[1].barh(df['driver'], df['avg_steering'], color=colours)
    axes[1].set_title('Avg Steering (lower = smoother)', fontweight='bold')
    axes[1].invert_yaxis()

    # Smoothness score (higher = better)
    axes[2].barh(df['driver'], df['smoothness'], color=colours)
    axes[2].set_title('Smoothness Score (higher = better)', fontweight='bold')
    axes[2].invert_yaxis()

    for ax in axes:
        ax.spines[['top', 'right']].set_visible(False)

    plt.suptitle('🏆 Box Box Bot — Driver Comparison', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('replays/driver_comparison.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('📸 Saved to replays/driver_comparison.png')

    print('\n🏆 Standings:')
    print(df[['driver', 'total_reward', 'avg_steering', 'smoothness']].to_string(index=False))

---
## 📺 11 — View Replay GIFs

Display replay GIFs for all trained drivers side by side.

Run after `python agents/evaluate.py`.

In [ ]:
CHECKPOINT = 1_000_000  # ← change to view a different checkpoint

for name, cfg in F1DriverEnv.PERSONALITIES.items():
    gif_path = f'replays/{name}_{CHECKPOINT}.gif'
    if os.path.exists(gif_path):
        print(f"\n{cfg['emoji']}  #{cfg['number']}  {name.replace('_', ' ').title()}")
        print(f"   {cfg['description']}")
        display(Image(filename=gif_path))
    else:
        print(f'  ⚠️  No GIF found for {name} @ {CHECKPOINT:,} steps — run evaluate.py first.')

---
## 🏁 What's Next?

```bash
# 1. Test the full pipeline (quick — 5k steps)
python agents/train.py --test

# 2. Full championship training (~3–4 hours)
python agents/train.py

# 3. Record GIFs and standings
python agents/evaluate.py

# 4. Come back to cells 10 and 11 above to visualise results
```

> *"We are checking the data. The agent is, uh... we are checking."*